# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Sonija/Flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2: Content Refresh / Opportunity Scoring**

This notebook builds, evaluates, and compares supervised machine learning models against our frozen Week 4 rule baseline for prioritising content refreshes. It enforces an honest 5-fold `GroupKFold` cross-validation split by client, evaluates Precision@K on candidate queues, calculates permutation feature importances, and performs error analysis on false positive and false negative failure modes.


## 1. Method choice and why

### Model Suite Progression
To answer our **Lane 2** decision question (*"Which pages should an editor review first for a content refresh?"*), we evaluate a progression of models ranging from simple interpretable baselines to flexible tree ensembles:

1. **Week 4 Frozen Rule Baseline**: Transparent, hand-written policy (`score = stale_flag * visible_flag * impressions_90d`). No fitted weights.
2. **Logistic Regression (Linear Baseline)**: L2-penalized, standardized linear model. Provides readable feature coefficients to test linear signal relationships (e.g. negative impact of content age vs positive impact of impression volume).
3. **Decision Tree Classifier (`max_depth=4`)**: Shallow decision tree that learns explicit multi-condition split thresholds. Readable out loud to non-technical stakeholders.
4. **Random Forest Classifier (`n_estimators=100`, `max_depth=10`)**: Ensemble of randomized decision trees. Captures non-linear feature interactions without overfitting to individual client quirks.
5. **HistGradientBoostingClassifier (`max_depth=6`)**: Gradient boosted decision trees. Iteratively fits small trees on residual errors, providing strong predictive power on tabular data.

### Safe Feature Set & Exclusions
All model features are strictly knowable **prior** to the decision point:
- **Features Used**: `impressions_90d`, `clicks_90d`, `avg_position` (0 filled with 50.0), `days_since_last_update` (filled with `content_age_days`), `days_with_impressions`, `ctr`, `engagement_rate`, `content_age_days`.
- **Excluded (Preventing Data Leakage)**: `trend_direction` and `trend_pct` (used solely to construct the ground-truth label `is_declining_label`), `content_id` and `client_id` (identifiers used for grouping/splitting only).


In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Ground-truth binary target derived from trend_direction (down = 1)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

base_rate = df['is_declining_label'].mean()
print(f"Total Content Items Analyzed: {len(df):,}")
print(f"Distinct Pseudonymized Clients: {len(df['client_id'].unique())}")
print(f"Dataset Overall Base Rate (% declining): {base_rate:.2%}")

# Define safe pre-decision feature set
feature_cols = [
    'impressions_90d',
    'clicks_90d',
    'avg_position',
    'days_since_last_update',
    'days_with_impressions',
    'ctr',
    'engagement_rate',
    'content_age_days'
]

X = df[feature_cols].copy()
X['avg_position'] = np.where(X['avg_position'] == 0, 50.0, X['avg_position'])
X['engagement_rate'] = X['engagement_rate'].fillna(0.0)
X['ctr'] = X['ctr'].fillna(0.0)
X['days_since_last_update'] = X['days_since_last_update'].fillna(X['content_age_days'])

y = df['is_declining_label'].values
groups = df['client_id'].values

print(f"Feature matrix shape: {X.shape}")
print("Features declared safe and leak-free:", feature_cols)


Total Content Items Analyzed: 30,000
Distinct Pseudonymized Clients: 32
Dataset Overall Base Rate (% declining): 54.21%
Feature matrix shape: (30000, 8)
Features declared safe and leak-free: ['impressions_90d', 'clicks_90d', 'avg_position', 'days_since_last_update', 'days_with_impressions', 'ctr', 'engagement_rate', 'content_age_days']


## 2. Split design

### Honest Grouped Validation (`GroupKFold` by `client_id`)
In real-world deployment, FlyRank's refresh scoring system must evaluate new pages for **unseen clients**. 

As demonstrated in scikit-learn best practices and Week 5 lectures, a standard **Random Split** creates severe data leakage: pages from the same client appear in both training and test sets. Tree ensembles easily memorize client identity (e.g. "this client drops 30% of the time") rather than learning general content decay signals.

Below, we compare **Random Split (5-Fold KFold)** vs. **Grouped Split (5-Fold GroupKFold by `client_id`)** to prove why random splitting yields deceptively inflated numbers.


In [2]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestClassifier

def eval_queue_p_at_k(scores, labels, impressions, k_list=[10, 20, 50, 100]):
    order = np.lexsort((-impressions, -scores))
    sorted_labels = labels[order]
    return {f'P@{k}': sorted_labels[:k].mean() for k in k_list}

# 1. Random Split Evaluation
kf_rand = KFold(n_splits=5, shuffle=True, random_state=42)
rand_p50_scores = []
for tr, te in kf_rand.split(X, y):
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X.iloc[tr], y[tr])
    probs = rf.predict_proba(X.iloc[te])[:, 1]
    p50 = eval_queue_p_at_k(probs, y[te], df.iloc[te]['impressions_90d'].values)['P@50']
    rand_p50_scores.append(p50)

# 2. Grouped Split Evaluation (GroupKFold by client_id)
gkf_group = GroupKFold(n_splits=5)
group_p50_scores = []
for tr, te in gkf_group.split(X, y, groups):
    # Verify zero client leakage between train and test folds
    train_clients = set(groups[tr])
    test_clients = set(groups[te])
    assert len(train_clients & test_clients) == 0, "CLIENT LEAKAGE DETECTED!"
    
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    rf.fit(X.iloc[tr], y[tr])
    probs = rf.predict_proba(X.iloc[te])[:, 1]
    p50 = eval_queue_p_at_k(probs, y[te], df.iloc[te]['impressions_90d'].values)['P@50']
    group_p50_scores.append(p50)

print("=== SPLIT DESIGN COMPARISON ===")
print(f"Random Split RF Mean P@50 : {np.mean(rand_p50_scores):.2%} ± {np.std(rand_p50_scores):.2%}")
print(f"Grouped Split RF Mean P@50: {np.mean(group_p50_scores):.2%} ± {np.std(group_p50_scores):.2%}")
print("\nDiagnostic Finding: Random split inflates evaluation by ~13 percentage points due to client memorization. GroupKFold is strictly enforced for all model comparisons.")


=== SPLIT DESIGN COMPARISON ===
Random Split RF Mean P@50 : 90.80% ± 3.49%
Grouped Split RF Mean P@50: 77.60% ± 5.71%

Diagnostic Finding: Random split inflates evaluation by ~13 percentage points due to client memorization. GroupKFold is strictly enforced for all model comparisons.


## 3. Train + compare vs my baseline

### The Non-Negotiable Comparison Table
We evaluate all models on identical 5-fold `GroupKFold` client-heldout splits on candidate queues sorted by predicted probability (breaking ties with `impressions_90d` descending).

The table reports mean $\pm$ standard deviation across folds for Precision@K (P@10, P@20, P@50, P@100), PR-AUC, ROC-AUC, and the dataset Base Rate (**54.21%**). Summary receipts are saved to `work/outputs/model_metrics.json`.


In [3]:
import json
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

# Week 4 Frozen Baseline Rule score
rule_scores = ((df['days_since_last_update'] >= 90) & (df['avg_position'] > 0) & (df['avg_position'] <= 30)).astype(int) * df['impressions_90d']

model_dict = {
    'Baseline Rule (W4)': 'rule',
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)),
    'Decision Tree (depth=4)': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest (n=100)': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_depth=6, random_state=42)
}

suite_results = {m: {col: [] for col in ['P@10', 'P@20', 'P@50', 'P@100', 'PR-AUC', 'ROC-AUC']} for m in model_dict}

for fold, (tr, te) in enumerate(gkf_group.split(X, y, groups)):
    X_tr, y_tr = X.iloc[tr], y[tr]
    X_te, y_te = X.iloc[te], y[te]
    imp_te = df.iloc[te]['impressions_90d'].values
    
    # Baseline Rule
    r_score_te = rule_scores.iloc[te].values
    r_pk = eval_queue_p_at_k(r_score_te, y_te, imp_te)
    for k_str, v in r_pk.items():
        suite_results['Baseline Rule (W4)'][k_str].append(v)
    suite_results['Baseline Rule (W4)']['PR-AUC'].append(average_precision_score(y_te, r_score_te))
    suite_results['Baseline Rule (W4)']['ROC-AUC'].append(roc_auc_score(y_te, r_score_te))
    
    # Supervised Models
    for name, model in model_dict.items():
        if name == 'Baseline Rule (W4)':
            continue
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_te)[:, 1]
        m_pk = eval_queue_p_at_k(probs, y_te, imp_te)
        for k_str, v in m_pk.items():
            suite_results[name][k_str].append(v)
        suite_results[name]['PR-AUC'].append(average_precision_score(y_te, probs))
        suite_results[name]['ROC-AUC'].append(roc_auc_score(y_te, probs))

# Build summary comparison table
summary_rows = []
for name, res in suite_results.items():
    row = {'Model': name}
    for col in ['P@10', 'P@20', 'P@50', 'P@100', 'PR-AUC', 'ROC-AUC']:
        arr = res[col]
        row[col] = f"{np.mean(arr):.2%} ± {np.std(arr):.2%}"
    summary_rows.append(row)

comp_df = pd.DataFrame(summary_rows)
display(comp_df) if 'display' in globals() else print(comp_df.to_string(index=False))

# Export summary metrics JSON
os.makedirs('../outputs', exist_ok=True)
json_export = {
    "base_rate": base_rate,
    "split_type": "5-Fold GroupKFold by client_id",
    "models": {}
}

for name, res in suite_results.items():
    json_export["models"][name] = {
        col: {"mean": float(np.mean(res[col])), "std": float(np.std(res[col]))}
        for col in ['P@10', 'P@20', 'P@50', 'P@100', 'PR-AUC', 'ROC-AUC']
    }

metrics_json_path = '../outputs/model_metrics.json'
with open(metrics_json_path, 'w') as f:
    json.dump(json_export, f, indent=2)

print(f"\nSuccessfully exported metrics receipts to {metrics_json_path}")


                  Model            P@10            P@20            P@50           P@100         PR-AUC        ROC-AUC
     Baseline Rule (W4) 48.00% ± 13.27% 46.00% ± 13.56% 45.20% ± 14.62% 48.60% ± 18.21% 55.42% ± 9.75% 52.62% ± 4.10%
    Logistic Regression 82.00% ± 17.20% 83.00% ± 12.49%  83.60% ± 9.33%  80.40% ± 5.71% 66.99% ± 6.06% 64.79% ± 6.27%
Decision Tree (depth=4)  62.00% ± 9.80%  60.00% ± 7.75% 64.80% ± 12.56% 69.20% ± 13.48% 64.28% ± 7.63% 64.85% ± 6.03%
  Random Forest (n=100) 86.00% ± 13.56% 82.00% ± 11.66%  77.60% ± 5.71%  78.60% ± 4.76% 69.58% ± 5.96% 68.57% ± 4.99%
   HistGradientBoosting  90.00% ± 6.32%  90.00% ± 3.16%  82.40% ± 7.31%  82.60% ± 6.47% 69.30% ± 6.59% 68.56% ± 4.74%

Successfully exported metrics receipts to ../outputs/model_metrics.json


## 4. Errors and interpretation

### Feature Importances & Permutation Importance
We calculate Gini Feature Importances and Permutation Importances on held-out test folds to observe what signals drive tree ensemble predictions:
- **`avg_position`** (+0.0646 permutation importance) and **`content_age_days`** (+0.0281) emerge as the strongest predictive signals for decline risk.
- High search volume (`impressions_90d`, `clicks_90d`) provides essential scale context.

---

### Tie-Breaking Policy
When candidate items produce identical model probabilities (especially in shallow Decision Trees), ties are broken deterministically by sorting by `impressions_90d` descending, ensuring that higher-visibility opportunities are consistently prioritized.

---

### Error Analysis: Concrete Case Studies

1. **Top False Positives (Model predicted high decline risk, but page remained stable)**:
   - *Example (`content_d144`)*: Prob = 0.86, Position = 4.4, Stale = 104d, CTR = 0.00%, `is_declining_label = 0` (Stable).
   - *Failure Mode*: High search volume with 0.00% CTR and staleness triggered high decline probability, but search impression volume remained flat. The failure mode is snippet CTR capture (`SNIPPET_FIX`), not article body decay.
2. **Top False Negatives (Model predicted low decline risk, but page declined)**:
   - *Example (`content_a4ad`)*: Prob = 0.13, Position = 51.6, Stale = 22d, `is_declining_label = 1` (Declined).
   - *Failure Mode*: Model assumed fresh content (22 days old) was safe, but deep position pages (> 50) can experience rapid impression drops due to search algorithm re-indexing regardless of recency.


In [4]:
from sklearn.inspection import permutation_importance

# Train Random Forest on fold 1 for feature importance & error inspection
tr_1, te_1 = next(gkf_group.split(X, y, groups))
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X.iloc[tr_1], y[tr_1])

perm_imp = permutation_importance(rf_model, X.iloc[te_1], y[te_1], n_repeats=5, random_state=42)
imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Gini Importance': rf_model.feature_importances_,
    'Permutation Importance Mean': perm_imp.importances_mean,
    'Permutation Importance Std': perm_imp.importances_std
}).sort_values(by='Permutation Importance Mean', ascending=False)

print("=== RANDOM FOREST FEATURE IMPORTANCE ===")
display(imp_df) if 'display' in globals() else print(imp_df.to_string(index=False))

# Error Analysis
probs_te1 = rf_model.predict_proba(X.iloc[te_1])[:, 1]
te_df = df.iloc[te_1].copy()
te_df['predicted_prob'] = probs_te1
te_df = te_df.sort_values(by=['predicted_prob', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)

fps = te_df[te_df['is_declining_label'] == 0].head(3)
fns = te_df[te_df['is_declining_label'] == 1].tail(3)

print("\n=== TOP 3 FALSE POSITIVES (Model predicted high decline risk, but page remained stable) ===")
for i, r in fps.iterrows():
    print(f"ID={r['content_id'][:12]} | Client={r['client_id'][:10]} | Prob={r['predicted_prob']:.2f} | Imp={r['impressions_90d']:,} | Pos={r['avg_position']:.1f} | Stale={r['days_since_last_update']}d | CTR={r['ctr']:.2f}% | Declining={r['is_declining_label']}")

print("\n=== TOP 3 FALSE NEGATIVES (Model predicted low decline risk, but page declined) ===")
for i, r in fns.iterrows():
    print(f"ID={r['content_id'][:12]} | Client={r['client_id'][:10]} | Prob={r['predicted_prob']:.2f} | Imp={r['impressions_90d']:,} | Pos={r['avg_position']:.1f} | Stale={r['days_since_last_update']}d | CTR={r['ctr']:.2f}% | Declining={r['is_declining_label']}")


=== RANDOM FOREST FEATURE IMPORTANCE ===
               Feature  Gini Importance  Permutation Importance Mean  Permutation Importance Std
          avg_position         0.175631                     0.064640                    0.001369
      content_age_days         0.159500                     0.028054                    0.003501
            clicks_90d         0.057694                     0.018350                    0.003559
       impressions_90d         0.207456                     0.012043                    0.002539
days_since_last_update         0.067003                     0.004224                    0.002291
                   ctr         0.060608                     0.003710                    0.002995
       engagement_rate         0.027216                     0.003682                    0.001066
 days_with_impressions         0.244892                     0.001170                    0.002104

=== TOP 3 FALSE POSITIVES (Model predicted high decline risk, but page remained stabl

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
